Creating a fasta which isolated Antibody Sequences (not including antigens) and displays the species of the targeted antigen

In [6]:
from Bio import SeqIO

for i, record in enumerate(SeqIO.parse("pdb_sequences.fasta", "fasta")):
    print(record.id)
    if i == 10:
        break

import os
print(os.getcwd())


7ZOZ_1|A|Homo
7ZOZ_2|H|Homo
7ZOZ_3|L|Homo
6PE8_1|A,H|Homo
6PE8_2|B,L|Homo
6PE8_3|T,U|Homo
8IVX_1|A|Homo
8IVX_2|H|Homo
8IVX_3|L|Homo
6VMK_1|C,F,I,N,Q,T,W,X,a,d,g,j,m,p,s,v|Homo
6VMK_2|A,D,G,J,L,O,R,U,Y,b,e,h,k,n,q,t|Homo
/home/utzer/new_temp/group04-team03/blasting/Multiple Sequence Alignment


In [7]:
import csv
from Bio import SeqIO

# === CONFIG ===
sabdab_tsv = "../domain_analysis/data/sabdab_summary_all.tsv"
input_fasta = "pdb_sequences.fasta"
output_fasta = "antibody_only.fasta"

# === STEP 1: Build set of antibody chains from SAbDab ===
antibody_chains = set()

with open(sabdab_tsv, newline='', encoding='utf-8') as tsvfile:
    reader = csv.DictReader(tsvfile, delimiter='\t')
    for row in reader:
        pdb = row["pdb"].upper()
        for chain in row["Hchain"].split(",") + row["Lchain"].split(","):
            chain = chain.strip()
            if chain:
                antibody_chains.add(f"{pdb}|{chain.upper()}")

# === STEP 2: Parse FASTA and check for antibody chains ===
kept_records = []

for record in SeqIO.parse(input_fasta, "fasta"):
    header_parts = record.id.split("_")  # e.g., 6PE8_1|A,H|Homo → ["6PE8", "1|A,H|Homo"]
    pdb_id = header_parts[0].upper()

    if "|" in record.id:
        chain_info = record.id.split("|")[1]  # e.g., "A,H"
        chains = [c.strip().upper() for c in chain_info.split(",")]
    else:
        chains = []

    # Check if any chain matches an antibody chain
    for chain in chains:
        if f"{pdb_id}|{chain}" in antibody_chains:
            kept_records.append(record)
            break  # Keep this record if *any* chain is a match

# === STEP 3: Save result ===
SeqIO.write(kept_records, output_fasta, "fasta")
print(f"✅ Kept {len(kept_records)} records with antibody chains.")



✅ Kept 2818 records with antibody chains.


In [9]:
import csv
from Bio import SeqIO

# Path to your metadata file (CSV or TSV)
metadata_file = "../../data_cleanup/df_sars_hum.csv"


# Dictionary to store metadata keyed by (pdb_id, chain)
metadata = {}

# Read t
with open(metadata_file, newline='') as file:
    reader = csv.DictReader(file, delimiter='\t')  # Use '\t' if TSV
    
    for row in reader:
        pdb_id = row['pdb'].lower()
        antigen_species = row['antigen_species'].lower()
        hchain = row['Hchain'].strip()
        lchain = row['Lchain'].strip()

        if hchain:
            metadata[(pdb_id, hchain)] = antigen_species
        if lchain:
            metadata[(pdb_id, lchain)] = antigen_species

# Input and output FASTA file paths
input_fasta = "antibody_only.fasta"
output_fasta = "annotated_antibody_only.fasta"

# Open output file for writing annotated sequences
with open(output_fasta, 'w') as out_handle:
    for record in SeqIO.parse(input_fasta, "fasta"):
        header = record.description
        parts = header.split("|")
        pdb_chain = parts[0].lower().split('_')[0]
        chain_str = parts[1].strip()
        chains = [c.strip() for c in chain_str.split(",")]

        # Get antigen species for all chains in this record
        antigen_list = [metadata.get((pdb_chain, chain), "unknown") for chain in chains]
        antigen_combined = ",".join(sorted(set(antigen_list)))

        # Construct new header with annotation
        new_header = f"{header} | antigen_species={antigen_combined}"

        # Write to output FASTA file
        out_handle.write(f">{new_header}\n")
        out_handle.write(str(record.seq) + "\n")


In [10]:
# Count occurrences of full species names in annotated FASTA

species_to_count = {
    "severe acute respiratory syndrome coronavirus2": 0,
    "homo sapiens": 0,
}

annotated_fasta = "annotated_antibody_only.fasta"

with open(annotated_fasta, 'r') as f:
    for line in f:
        if line.startswith('>'):
            if "antigen_species=" in line:
                antigens_part = line.split("antigen_species=")[1].strip()
                species_list = [s.strip().lower() for s in antigens_part.split(",")]
                for sp in species_to_count:
                    if sp in species_list:
                        species_to_count[sp] += 1

print("Counts of antigen species in annotated FASTA:")
for species, count in species_to_count.items():
    print(f"{species}: {count}")


Counts of antigen species in annotated FASTA:
severe acute respiratory syndrome coronavirus2: 1732
homo sapiens: 929
